# Q3: Text Summarization

**Text summarization** is the task of producing a concise, fluent, and informationally faithful condensation of a longer source document. This notebook compares two paradigmatically different summarization approaches on the **CNN/DailyMail** newswire dataset:

| # | System | Paradigm | Core Mechanism |
|---|---|---|---|
| 1 | **TextRank** | Extractive | Graph-based sentence ranking; selects verbatim sentences |
| 2 | **BART** (`facebook/bart-large-cnn`) | Abstractive | Sequence-to-sequence generation; produces novel text |

**Extractive** methods preserve factual accuracy by copying source sentences but cannot rephrase, compress, or merge information. **Abstractive** methods can generate more fluent and concise output but risk introducing hallucinated or unfaithful content.

Evaluation is performed using a multi-metric suite: **ROUGE-1/2/L** (lexical overlap), **BLEU** (n-gram precision with brevity penalty), **METEOR** (unigram alignment with stemming and synonymy), and **BERTScore** (contextual semantic similarity). Quantitative results are supplemented by qualitative inspection of individual examples.

## 1. Environment Setup

The following libraries are required and installed in this cell:

- `datasets` — HuggingFace library for loading CNN/DailyMail and other NLP benchmarks
- `transformers` — pre-trained model loading, tokenization, and generation for BART
- `evaluate` — standardized metric computation (ROUGE, BLEU, METEOR, BERTScore)
- `rouge_score` — backend for ROUGE computation
- `sacrebleu` — reproducible, detokenized BLEU scoring
- `bert-score` — contextual embedding-based similarity metric
- `nltk` — sentence tokenization (`sent_tokenize`) used inside TextRank
- `scikit-learn` — TF-IDF vectorization and cosine similarity for TextRank graph construction
- `networkx` — graph construction and PageRank computation for TextRank
- `accelerate` — enables efficient BART inference on GPU when available

A global random seed of **42** is fixed across Python, NumPy, and PyTorch to ensure reproducibility of the dataset subset selection and any stochastic operations.

In [1]:
!pip -q install datasets transformers evaluate rouge_score sacrebleu bert-score nltk scikit-learn networkx accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.2 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [2]:
# ============================================================
# Step 1: Imports, reproducibility, and environment check
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import pipeline

import nltk
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")

import evaluate

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python packages imported successfully.")
print("Random seed:", SEED)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")

[nltk_data] Downloading package punkt to /Users/sedaguven/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sedaguven/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/sedaguven/nltk_data...


Python packages imported successfully.
Random seed: 42
CUDA available: False
No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.


## 2. Dataset: CNN/DailyMail

The **CNN/DailyMail dataset** (Hermann et al., 2015; Nallapati et al., 2016) is the dominant benchmark for news summarization. It contains over 300,000 English newswire articles paired with human-written reference summaries (referred to as "highlights") extracted from the original news websites.

Version **3.0.0** is the standard non-anonymized version used in most contemporary summarization research. Its key properties:

| Property | Value |
|---|---|
| Training examples | ~287,000 |
| Validation examples | ~13,000 |
| Test examples | ~11,500 |
| Average article length | ~700 words |
| Average summary length | ~53 words |
| Compression ratio | ~13:1 |

Due to computational constraints, a **reproducible random subset of 20 test examples** is used for evaluation (`seed=42`). This subset is drawn from the test split — the same split used by published BART results — so comparisons with reported scores remain meaningful in terms of data origin, though aggregate numbers will differ due to the small subset size.

In [3]:
# ============================================================
# Step 2: Load CNN/DailyMail dataset
# ============================================================

from datasets import load_dataset

# CNN/DailyMail has several versions. Version 3.0.0 is the standard one used in many summarization experiments.
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)
print("\nColumn names:")
print(dataset["train"].column_names)

print("\nExample keys:")
print(dataset["train"][0].keys())

print("\nSample article:")
print(dataset["train"][0]["article"][:1000])

print("\nSample reference summary / highlights:")
print(dataset["train"][0]["highlights"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

Column names:
['article', 'highlights', 'id']

Example keys:
dict_keys(['article', 'highlights', 'id'])

Sample article:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a 

In [4]:
# ============================================================
# Step 2.1: Create a reproducible subset
# ============================================================

SUBSET_SIZE = 20

test_subset = dataset["test"].shuffle(seed=SEED).select(range(SUBSET_SIZE))

print("Subset size:", len(test_subset))
print("Columns:", test_subset.column_names)

# Convert to pandas for easier inspection
df = pd.DataFrame(test_subset)

df.head()

Subset size: 20
Columns: ['article', 'highlights', 'id']


,article,highlights,id
0,(CNN) I see signs of a revolution everywhere. ...,CNN's Dr. Sanjay Gupta says we should legalize...,12078b09d95c01cedb06da7fc63faab540432dee
1,He looks barely teenage. But this child has am...,Child has amassed thousands of Twitter followe...,5c3dd1296dfb1ecb54a188e0532394aed98c5c5b
2,New Jersey Governor Chris Christie wasn't look...,The presidential hopeful held a town hall meet...,ba1436809e689c3fdb97f9da8efd404f4d070920
3,YouTube star Cassey Ho has hit back at critics...,Cassey Ho boasts over two million subscribers ...,f843fb66ec425eb58453494ff8f838d390b243f3
4,British taekwondo fighter Aaron Cook has confi...,Aaron Cook was overlooked by Team GB for the L...,1872c284053c494149c6abca7ee6ddeb7a540335


In [6]:
# ============================================================
# Step 2.2: Basic dataset statistics
# ============================================================

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize

df["article_word_count"] = df["article"].apply(lambda x: len(x.split()))
df["summary_word_count"] = df["highlights"].apply(lambda x: len(x.split()))
df["article_sentence_count"] = df["article"].apply(lambda x: len(sent_tokenize(x)))
df["summary_sentence_count"] = df["highlights"].apply(lambda x: len(sent_tokenize(x)))

stats = df[
    [
        "article_word_count",
        "summary_word_count",
        "article_sentence_count",
        "summary_sentence_count",
    ]
].describe()

print(stats)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


       article_word_count  summary_word_count  article_sentence_count  \
count           20.000000           20.000000                20.00000   
mean           739.150000           51.450000                37.85000   
std            352.371257           22.605251                21.27582   
min            294.000000           26.000000                14.00000   
25%            489.250000           36.000000                24.25000   
50%            673.000000           45.000000                30.00000   
75%            946.250000           61.000000                48.25000   
max           1430.000000          107.000000                88.00000   

       summary_sentence_count  
count               20.000000  
mean                 3.500000  
std                  1.357242  
min                  2.000000  
25%                  2.750000  
50%                  3.000000  
75%                  4.000000  
max                  7.000000  


## 3. Extractive Summarization: TextRank

**TextRank** (Mihalcea & Tarau, 2004) is an unsupervised graph-based algorithm adapted from Google's PageRank for sentence ranking. It requires no training data and produces summaries by selecting the most "central" sentences from the source article.

**Algorithm steps:**
1. **Sentence segmentation** — the article is split into sentences using NLTK's `sent_tokenize`; very short sentences (fewer than 8 words) are filtered out to avoid fragments, captions, or rhetorical questions dominating the output
2. **TF-IDF representation** — each sentence is represented as a TF-IDF vector over the document vocabulary, capturing its lexical content
3. **Similarity graph construction** — a fully connected undirected graph is built where nodes are sentences and edge weights are the pairwise cosine similarity between their TF-IDF vectors
4. **PageRank scoring** — the NetworkX PageRank algorithm assigns an importance score to each sentence based on how similar it is to all other sentences; sentences that are highly similar to many others are considered central to the document's topic
5. **Sentence selection** — the top-3 sentences by PageRank score are selected and returned **in their original article order** to preserve narrative coherence

**Limitations**: TextRank cannot merge information across sentences, cannot rephrase, and may select redundant sentences if the similarity graph has many near-duplicate nodes. It also entirely ignores any notion of salience beyond intra-document term co-occurrence.

In [9]:
# ============================================================
# 3. Extractive Summarization with TextRank
# ============================================================

def textrank_summarize(article, num_sentences=3, min_sentence_words=8):
    """
    Extractive summarization using TextRank.

    This version filters out very short sentences because short fragments,
    quotations, or rhetorical questions can receive high graph centrality
    but produce weak summaries.

    Steps:
    1. Split the article into sentences.
    2. Remove very short sentences.
    3. Represent remaining sentences using TF-IDF.
    4. Compute cosine similarity between sentence vectors.
    5. Build a sentence similarity graph.
    6. Run PageRank to score sentence importance.
    7. Select top-ranked sentences and restore original article order.
    """

    original_sentences = sent_tokenize(article)

    # Keep sentence index so we can restore original order later.
    indexed_sentences = [
        (idx, sent)
        for idx, sent in enumerate(original_sentences)
        if len(sent.split()) >= min_sentence_words
    ]

    # Fallback: if filtering removes too many sentences, use original sentences.
    if len(indexed_sentences) <= num_sentences:
        return " ".join(original_sentences[:num_sentences])

    filtered_indices = [idx for idx, sent in indexed_sentences]
    filtered_sentences = [sent for idx, sent in indexed_sentences]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000
    )

    tfidf_matrix = vectorizer.fit_transform(filtered_sentences)

    similarity_matrix = cosine_similarity(tfidf_matrix)
    np.fill_diagonal(similarity_matrix, 0.0)

    graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(graph, weight="weight")

    ranked_filtered_indices = sorted(
        scores.keys(),
        key=lambda idx: scores[idx],
        reverse=True
    )

    selected_filtered_indices = ranked_filtered_indices[:num_sentences]

    # Convert filtered sentence positions back to original article positions.
    selected_original_indices = sorted(
        [filtered_indices[i] for i in selected_filtered_indices]
    )

    summary = " ".join([original_sentences[idx] for idx in selected_original_indices])

    return summary

In [10]:
# ============================================================
# 3.1 Test TextRank on one example
# ============================================================

sample_idx = 0

sample_article = df.loc[sample_idx, "article"]
sample_reference = df.loc[sample_idx, "highlights"]

sample_extractive_summary = textrank_summarize(
    sample_article,
    num_sentences=3,
    min_sentence_words=8
)

print("ARTICLE PREVIEW:")
print(sample_article[:1500])

print("\nREFERENCE SUMMARY:")
print(sample_reference)

print("\nTEXTRANK EXTRACTIVE SUMMARY:")
print(sample_extractive_summary)

ARTICLE PREVIEW:
(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I see it in the new surgeon general who cites data showing just how helpful it can be. I see a revolution in the attitudes of everyday Americans. For the first time a majority, 53%, favor its legalization, with 77% supporting it for medical purposes. Support for legalization has risen 11 points in the past few years alone. In 1969, the first time Pew asked the question about legalization, only 12% of the nation was in favor. I see a revolution that is burning white hot among young people, but also shows up among the parents and 

## 4. Abstractive Summarization: BART

**BART** (Lewis et al., 2019) is a denoising autoencoder pre-trained with a corrupted-document reconstruction objective. Its architecture combines a bidirectional encoder (like BERT) with a left-to-right autoregressive decoder (like GPT), making it particularly effective for text generation tasks.

The checkpoint used here — `facebook/bart-large-cnn` — is the BART-large model (406M parameters) **fine-tuned specifically on CNN/DailyMail**, making it the most directly applicable off-the-shelf abstractive summarizer for this dataset.

**Key generation parameters:**

| Parameter | Value | Rationale |
|---|---|---|
| `max_input_tokens` | 1024 | Hard limit of BART's positional embeddings; long articles are truncated |
| `max_summary_tokens` | 80 | Prevents over-generation; CNN/DM highlights average ~53 words |
| `min_summary_tokens` | 30 | Prevents degenerate short outputs |
| `num_beams` | 4 | Beam search decodes 4 candidates and returns the highest-probability sequence |

**Truncation note**: CNN/DailyMail articles frequently exceed 1024 tokens. Truncation means the tail of an article may be excluded, which can cause the model to miss conclusions or later-mentioned entities. This is a known limitation of applying fixed-window transformer models to long documents.

In [12]:
# ============================================================
# 4. Abstractive Summarization with BART
# ============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

BART_MODEL_NAME = "facebook/bart-large-cnn"

bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL_NAME)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL_NAME)

bart_model = bart_model.to(device)
bart_model.eval()

print("BART model and tokenizer loaded successfully.")

Using device: cuda


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART model and tokenizer loaded successfully.


In [13]:
# ============================================================
# 4.1 BART Summarization Function
# ============================================================

def bart_summarize(
    article,
    max_input_tokens=1024,
    max_summary_tokens=80,
    min_summary_tokens=30,
    num_beams=4
):
    """
    Generate an abstractive summary using facebook/bart-large-cnn.

    BART has a maximum input length of 1024 tokens, so long CNN/DailyMail
    articles are truncated to fit the model input limit.
    """

    inputs = bart_tokenizer(
        article,
        max_length=max_input_tokens,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        summary_ids = bart_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_summary_tokens,
            min_length=min_summary_tokens,
            num_beams=num_beams,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    summary = bart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return summary

In [14]:
# ============================================================
# 4.2 Test BART on one example
# ============================================================

sample_idx = 0

sample_article = df.loc[sample_idx, "article"]
sample_reference = df.loc[sample_idx, "highlights"]

sample_extractive_summary = textrank_summarize(
    sample_article,
    num_sentences=3,
    min_sentence_words=8
)

sample_abstractive_summary = bart_summarize(
    sample_article,
    max_input_tokens=1024,
    max_summary_tokens=80,
    min_summary_tokens=30,
    num_beams=4
)

print("REFERENCE SUMMARY:")
print(sample_reference)

print("\nTEXTRANK EXTRACTIVE SUMMARY:")
print(sample_extractive_summary)

print("\nBART ABSTRACTIVE SUMMARY:")
print(sample_abstractive_summary)

REFERENCE SUMMARY:
CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"

TEXTRANK EXTRACTIVE SUMMARY:
I see this medical marijuana revolution in surprising places. Just two years later, in "Weed 3," we are eyewitnesses to a revolution in full swing. So, here it is: We should legalize medical marijuana.

BART ABSTRACTIVE SUMMARY:
CNN's John Sutter says he sees signs of a medical marijuana revolution everywhere. For the first time a majority, 53%, favor its legalization, he says. Support for legalization has risen 11 points in the past few years alone. Sutter: "Weed 3" is eyewitnesses to a revolution in full swing.


In [21]:
# ============================================================
# 4.3 Generate TextRank and BART summaries for all examples
# ============================================================

from tqdm.auto import tqdm
import time

extractive_summaries = []
abstractive_summaries = []

start_time = time.time()

for article in tqdm(df["article"], desc="Generating summaries"):
    ext_summary = textrank_summarize(
        article,
        num_sentences=3,
        min_sentence_words=8
    )

    abs_summary = bart_summarize(
        article,
        max_input_tokens=1024,
        max_summary_tokens=100,
        min_summary_tokens=30,
        num_beams=4
    )

    extractive_summaries.append(ext_summary)
    abstractive_summaries.append(abs_summary)

end_time = time.time()

df["textrank_summary"] = extractive_summaries
df["bart_summary"] = abstractive_summaries

generation_time_seconds = end_time - start_time

print("Generated summaries for", len(df), "examples.")
print("Total generation time in seconds:", round(generation_time_seconds, 2))
print("Average time per example in seconds:", round(generation_time_seconds / len(df), 2))

df[["highlights", "textrank_summary", "bart_summary"]].head()

Generating summaries:   0%|          | 0/20 [00:00<?, ?it/s]

Generated summaries for 20 examples.
Total generation time in seconds: 7.41
Average time per example in seconds: 0.37


,highlights,textrank_summary,bart_summary
0,"CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .\nHe says he knows how easy it is do nothing ""because I did nothing for too long""","I see this medical marijuana revolution in surprising places. Just two years later, in ""Weed 3,"" we are eyewitnesses to a revolution in full swing. So, here it is: We should legalize medical marijuana.","CNN's John Sutter says he sees signs of a medical marijuana revolution everywhere. For the first time a majority, 53%, favor its legalization, he says. Support for legalization has risen 11 points in the past few years alone. Sutter: ""Weed 3"" is eyewitnesses to a revolution in full swing."
1,Child has amassed thousands of Twitter followers with 'gang life' photos .\nIn one video he points gun at camera as adults look on unfazed .\nHis tweets have prompted backlash with calls for intervention .,"But this child has amassed thousands of Twitter followers with his pictorial updates of 'gang life'. Baby-faced: This little boy has amassed more than 3,000 followers on Twitter with pictures like these . In many pictures he is smoking suspicious substances, with captions such as 'High Life' Backlash: The boy, from Memphis, has prompted a wave of critics calling his stunts 'sad' In one video he laughs and points the gun at the camera in an apparent attempt to look menacing - as adults laugh in the background.","Baby-faced boy from Memphis, Tennessee, has amassed more than 3,000 followers on Twitter. In many pictures he is smoking suspicious substances, with captions such as 'High Life' Tweets include the phrases, 'I need a bad b****', 'f*** da police', and 'gang sh** n****' He has prompted a wave of critics calling his stunts'sad'"
2,"The presidential hopeful held a town hall meeting in Kenilworth on Tuesday .\nDuring the meeting, high school English teacher Kathy Mooney got up to ask the governor a question about pensions .\nShe asked why he didn't seek a higher legal settlement in a case with ExxonMobil that would have contributed to the state's pension system .\nChristie responded by repeatedly asking how much Mooney knew about the deal instead of answering her question .","New Jersey Governor Chris Christie wasn't looking too presidential Tuesday night when he got into a heated debate with a veteran teacher at a town hall meeting. Not being nice: New Jersey Gov Chris Christie (left) is being called a bully for the way he interacted with a teacher (Kathy Mooney, right) at a Tuesday night town hall meeting . Perhaps the reason why Wollmer and his union responded sharply to Christie's town hall meeting Tuesday night, is that he blamed the union for their role in the current pension system.","New Jersey Governor Chris Christie got into a heated debate with a teacher at a town hall meeting Tuesday night. Kathy Mooney, a high school English Teacher from Roselle Park, questioned Christie's motivations behind a $225million legal settlement with oil company ExxonMobil. 'He's always taken a very nasty and disrespectful tone with teachers and other individuals who dare to question him at these events,' Steve Wollmer of the NJ Education Association told NJ.com."
3,Cassey Ho boasts over two million subscribers on her YouTube channel Blogilates .\nThe 28-year-old receives hundreds of comments a day telling her that she needs to lose weight .,"YouTube star Cassey Ho has hit back at critics with a powerful and provocative new video, highlighting the cruel comments left by viewers of her fitness-focused clips who accuse the trim and toned online icon of being everything from ‘too fat’ to ‘ugly’ to ‘pudgy’. Fighting back: In her new video, Pilates instructor and YouTube star Cassey Ho combats body-shamers who comment on her videos . In a post on her Blogilates blog, Cassey said negative comments on her videos are nothing new, but the flood of nastiness has grown especially bad lately.","Pilates instructor Cassey Ho, 28, says the negative comments in

## 5. Evaluation Metrics

Summarization quality is measured using four complementary automatic metrics, each capturing a different aspect of similarity between the generated summary and the reference highlight:


| Metric | What It Measures | Sensitivity |
|---|---|---|
| **ROUGE-1** | Unigram recall/precision/F1 between candidate and reference | Lexical overlap at word level |
| **ROUGE-2** | Bigram overlap | Captures local word order and phrasing |
| **ROUGE-L** | Longest Common Subsequence F1 | Sentence-level fluency and structure |
| **BLEU** | n-gram precision (1–4) with a brevity penalty for short outputs | Precision-focused; penalizes hallucination |
| **METEOR** | Unigram alignment with stemming, synonym matching, and word order | More lenient than BLEU; rewards paraphrasing |
| **BERTScore** | Token-level cosine similarity using `roberta-large` contextual embeddings | Semantic faithfulness; metric for abstractive output |

**Metric interpretation notes:**
- ROUGE and BLEU favor **extractive** systems because they directly copy source text, which tends to overlap highly with reference summaries also drawn from the source
- METEOR is more robust to abstractive output due to its synonym and stemming components
- BERTScore is the most appropriate metric for comparing abstractive summaries, as it can recognize semantically equivalent paraphrases that share no surface-level tokens with the reference

All metrics are computed against the CNN/DailyMail **highlights** as the single reference. Results should be interpreted relative to each other (TextRank vs. BART) rather than as absolute quality scores, given the 20-example subset size.

In [16]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

references = df["highlights"].tolist()
textrank_predictions = df["textrank_summary"].tolist()
bart_predictions = df["bart_summary"].tolist()

print("Evaluation metrics loaded successfully.")
print("Number of references:", len(references))
print("Number of TextRank predictions:", len(textrank_predictions))
print("Number of BART predictions:", len(bart_predictions))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Evaluation metrics loaded successfully.
Number of references: 20
Number of TextRank predictions: 20
Number of BART predictions: 20


In [17]:
def compute_all_metrics(predictions, references, model_name):
    """
    Compute ROUGE, BLEU, METEOR, and BERTScore for a summarization system.
    """

    # ROUGE expects predictions and references as lists of strings.
    rouge_result = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    # sacreBLEU expects references as list of list:
    # each prediction has one or more reference summaries.
    bleu_result = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    meteor_result = meteor_metric.compute(
        predictions=predictions,
        references=references
    )

    bertscore_result = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="en"
    )

    result = {
        "model": model_name,
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"],
        "meteor": meteor_result["meteor"],
        "bertscore_precision": float(np.mean(bertscore_result["precision"])),
        "bertscore_recall": float(np.mean(bertscore_result["recall"])),
        "bertscore_f1": float(np.mean(bertscore_result["f1"])),
    }

    return result

In [23]:
references = df["highlights"].tolist()
textrank_predictions = df["textrank_summary"].tolist()
bart_predictions = df["bart_summary"].tolist()

textrank_results = compute_all_metrics(
    predictions=textrank_predictions,
    references=references,
    model_name="TextRank"
)

bart_results = compute_all_metrics(
    predictions=bart_predictions,
    references=references,
    model_name="BART"
)

results_df = pd.DataFrame([textrank_results, bart_results])

# Make numbers easier to read
results_df_rounded = results_df.copy()
numeric_columns = results_df_rounded.columns.drop("model")
results_df_rounded[numeric_columns] = results_df_rounded[numeric_columns].round(4)

results_df_rounded

,model,rouge1,rouge2,rougeL,bleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,TextRank,0.3449,0.1393,0.2333,8.6269,0.3404,0.8669,0.8706,0.8686
1,BART,0.4229,0.1669,0.2756,12.2576,0.3825,0.8809,0.8798,0.8802


## 6. Qualitative Analysis

Automatic metrics provide aggregate overlap statistics but cannot fully capture several dimensions of summarization quality that matter in practice. This section examines individual examples to assess properties that metrics systematically underrepresent:

| Quality Dimension | Description | Expected difference |
|---|---|---|
| **Factual consistency** | Does the summary contain only claims supported by the source article? | TextRank is guaranteed consistent (verbatim copy); BART may hallucinate entities or figures |
| **Coherence** | Does the summary read as a fluent, connected text? | TextRank may produce disjointed sentence collections; BART generates connected prose |
| **Redundancy** | Does the summary repeat the same information? | TextRank may select thematically similar sentences; BART decoder produces non-redundant output by design |
| **Information coverage** | Does the summary capture the most important information? | TextRank is limited to salient surface sentences; BART can compress and integrate multiple source clauses |
| **Abstractiveness** | Does the summary paraphrase rather than copy? | TextRank output is 100% extractive; BART output varies — may copy frequent phrases or generate novel text |

For each selected example, the source article preview, reference highlight, TextRank summary, and BART summary are displayed side-by-side. Three examples (indices 0, 1, 2) are inspected and the full comparison table is saved to `q3_qualitative_examples.csv`.

In [24]:
qualitative_indices = [0, 1, 2]

for idx in qualitative_indices:
    print("=" * 100)
    print(f"EXAMPLE {idx}")
    print("=" * 100)

    print("\nARTICLE PREVIEW:")
    print(df.loc[idx, "article"][:2000])

    print("\nREFERENCE SUMMARY:")
    print(df.loc[idx, "highlights"])

    print("\nTEXTRANK EXTRACTIVE SUMMARY:")
    print(df.loc[idx, "textrank_summary"])

    print("\nBART ABSTRACTIVE SUMMARY:")
    print(df.loc[idx, "bart_summary"])

    print("\n")

EXAMPLE 0

ARTICLE PREVIEW:
(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I see it in the new surgeon general who cites data showing just how helpful it can be. I see a revolution in the attitudes of everyday Americans. For the first time a majority, 53%, favor its legalization, with 77% supporting it for medical purposes. Support for legalization has risen 11 points in the past few years alone. In 1969, the first time Pew asked the question about legalization, only 12% of the nation was in favor. I see a revolution that is burning white hot among young people, but also shows up among the p

In [25]:
#  Qualitative Comparison Table

qualitative_df = df.loc[
    qualitative_indices,
    [
        "article",
        "highlights",
        "textrank_summary",
        "bart_summary"
    ]
].copy()

qualitative_df["article_preview"] = qualitative_df["article"].apply(
    lambda x: x[:700].replace("\n", " ") + "..."
)

qualitative_df = qualitative_df[
    [
        "article_preview",
        "highlights",
        "textrank_summary",
        "bart_summary"
    ]
]

pd.set_option("display.max_colwidth", 1000)

qualitative_df

,article_preview,highlights,textrank_summary,bart_summary
0,"(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I see it in the new surgeon general who cites data showing just how helpful it can be. I see a revolution in the attitudes of everyday Americans. For the first time a majority, 53%, favor its legalization, with 77% supporting it for medical purposes. ...","CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .\nHe says he knows how easy it is do nothing ""because I did nothing for too long""","I see this medical marijuana revolution in surprising places. Just two years later, in ""Weed 3,"" we are eyewitnesses to a revolution in full swing. So, here it is: We should legalize medical marijuana.","CNN's John Sutter says he sees signs of a medical marijuana revolution everywhere. For the first time a majority, 53%, favor its legalization, he says. Support for legalization has risen 11 points in the past few years alone. Sutter: ""Weed 3"" is eyewitnesses to a revolution in full swing."
1,"He looks barely teenage. But this child has amassed thousands of Twitter followers with his pictorial updates of 'gang life'. The baby-faced boy from Memphis, Tennessee, poses with guns, cash, and bags of what looks like marijuana. Scroll down for video . Baby-faced: This little boy has amassed more than 3,000 followers on Twitter with pictures like these . In many pictures he is smoking suspicious substances, with captions such as 'High Life' Backlash: The boy, from Memphis, has prompted a wave of critics calling his stunts 'sad' In one video he laughs and points the gun at the camera in an apparent attempt to look menacing - as adults laugh in the background. In others, he is pictured blow...",Child has amassed thousands of Twitter followers with 'gang life' photos .\nIn one video he points gun at camera as adults look on unfazed .\nHis tweets have prompted backlash with calls for intervention .,"But this child has amassed thousands of Twitter followers with his pictorial updates of 'gang life'. Baby-faced: This little boy has amassed more than 3,000 followers on Twitter with pictures like these . In many pictures he is smoking suspicious substances, with captions such as 'High Life' Backlash: The boy, from Memphis, has prompted a wave of critics calling his stunts 'sad' In one video he laughs and points the gun at the camera in an apparent attempt to look menacing - as adults laugh in the background.","Baby-faced boy from Memphis, Tennessee, has amassed more than 3,000 followers on Twitter. In many pictures he is smoking suspicious substances, with captions such as 'High Life' Tweets include the phrases, 'I need a bad b****', 'f*** da police', and 'gang sh** n****' He has prompted a wave of critics calling his stunts'sad'"
2,"New Jersey Governor Chris Christie wasn't looking too presidential Tuesday night when he got into a heated debate with a veteran teacher at a town hall meeting. And now the state's largest teacher's union is calling him out for his 'bullying' behavior. 'He's always taken a very nasty and disrespectful tone with teachers and other individuals who dare to question him at these events,' Steve Wollmer of the NJ Education Association told NJ.com. 'It's the one thing that never seems to change.' Scroll Down for Video . Not being nice: New Jersey Gov Chris Christie (left) is being called a bully for the way he interacted with a teacher (Kathy Mooney, right) at a Tuesday night town hall meeting . Th...","The presidential hopeful held a town hall meeting in Kenilworth on Tuesday .\nDuring the meeting, high school 

In [26]:
# 7. Final Results and Export


# Save quantitative metric results
results_df_rounded.to_csv("q3_summarization_metric_results.csv", index=False)

# Save full generated summaries
df[
    [
        "id",
        "article",
        "highlights",
        "textrank_summary",
        "bart_summary",
        "article_word_count",
        "summary_word_count",
        "article_sentence_count",
        "summary_sentence_count"
    ]
].to_csv("q3_generated_summaries.csv", index=False)

# Save qualitative examples separately
qualitative_indices = [0, 1, 2]

qualitative_export_df = df.loc[
    qualitative_indices,
    [
        "id",
        "article",
        "highlights",
        "textrank_summary",
        "bart_summary"
    ]
].copy()

qualitative_export_df.to_csv("q3_qualitative_examples.csv", index=False)

print("Saved files:")
print("- q3_summarization_metric_results.csv")
print("- q3_generated_summaries.csv")
print("- q3_qualitative_examples.csv")

Saved files:
- q3_summarization_metric_results.csv
- q3_generated_summaries.csv
- q3_qualitative_examples.csv


In [27]:
# 7.1 Final Experiment Summary

print("Question 3 experiment completed.")
print("Dataset: CNN/DailyMail 3.0.0")
print("Split: test")
print("Subset size:", len(df))
print("Random seed:", SEED)
print("Extractive method: TextRank with TF-IDF sentence similarity and PageRank")
print("Abstractive method: facebook/bart-large-cnn")
print("BART decoding: num_beams=4, max_input_tokens=1024, min_summary_tokens=30, max_summary_tokens=100")
print("Generation time in seconds:", round(generation_time_seconds, 2))
print("Average generation time per example:", round(generation_time_seconds / len(df), 2))
print("\nFinal metric results:")
display(results_df_rounded)

Question 3 experiment completed.
Dataset: CNN/DailyMail 3.0.0
Split: test
Subset size: 20
Random seed: 42
Extractive method: TextRank with TF-IDF sentence similarity and PageRank
Abstractive method: facebook/bart-large-cnn
BART decoding: num_beams=4, max_input_tokens=1024, min_summary_tokens=30, max_summary_tokens=100
Generation time in seconds: 7.41
Average generation time per example: 0.37

Final metric results:


,model,rouge1,rouge2,rougeL,bleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,TextRank,0.3449,0.1393,0.2333,8.6269,0.3404,0.8669,0.8706,0.8686
1,BART,0.4229,0.1669,0.2756,12.2576,0.3825,0.8809,0.8798,0.8802
